In [1]:
%load_ext autoreload
%autoreload 2
import os
import shutil
import hydra
from scipy.spatial.transform import Rotation as R
import pathlib
from pathlib import Path
import wandb
import torch
import tqdm
import dill
import json
import numpy as np
import datetime
import matplotlib.pyplot as plt
from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.env_runner.robomimic_image_runner_joint_space import RobomimicImageRunnerJointSpace
from diffusion_policy.policy.diffusion_unet_hybrid_image_policy import DiffusionUnetHybridImagePolicy

from hydra import compose, initialize
from omegaconf import OmegaConf

config_path = Path("diffusion_policy/config/train_diffusion_unet_hybrid_workspace.yaml")
DATASET_PATH = "/data/scene-rep/u/iyu/scene-jacobian-discovery/diff-policy/diffusion_policy/data/robomimic/datasets/lift/ph/image_abs.hdf5"
overrides = [
    # "config-name=train_robomimic_image_joint_space_workspace.yaml",
    "task=lift_image_abs_joint_space",
    f"task.env_runner.dataset_path={DATASET_PATH}",
    "task.env_runner.n_train=0",
    "task.env_runner.n_train_vis=0",
    "task.env_runner.n_test=1",
    "task.env_runner.n_test_vis=1",
]

OmegaConf.register_new_resolver("eval", eval, replace=True)
with initialize(version_base=None, config_path=str("../../" / config_path.parent)):
    cfg = compose(config_name=str(config_path.name), overrides=overrides)

OmegaConf.resolve(cfg)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ":1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_runner = hydra.utils.instantiate(cfg.task.env_runner, output_dir=None)

/home/iyu/miniconda3/envs/robodiff-orig/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env kwargs {'has_renderer': False, 'has_offscreen_renderer': True, 'ignore_done': True, 'use_object_obs': False, 'use_camera_obs': True, 'control_freq': 20, 'controller_configs': {'type': 'JOINT_POSITION', 'input_max': 1, 'input_min': -1, 'output_max': 0.05, 'output_min': -0.05, 'kp': 50, 'damping_ratio': 1, 'impedance_mode': 'fixed', 'kp_limits': [0, 300], 'damping_ratio_limits': [0, 10], 'qpos_limits': None, 'interpolation': None, 'ramp_ratio': 0.2}, 'robots': ['Panda'], 'camera_depths': False, 'camera_heights': 84, 'camera_widths': 84, 'reward_shaping': False, 'camera_names': ['agentview', 'robot0_eye_in_hand'], 'render_gpu_device_id': 0}


### Load the dataset

In [2]:
import h5py
import mediapy as media
f = h5py.File(DATASET_PATH, "r")
demos = sorted(list(f["data"].keys()))
# extract filter key information
if "mask" in f:
    all_filter_keys = {}
    for fk in f["mask"]:
        fk_demos = sorted([elem.decode("utf-8") for elem in np.array(f["mask/{}".format(fk)])])
        all_filter_keys[fk] = fk_demos

# put demonstration list in increasing episode order
inds = np.argsort([int(elem[5:]) for elem in demos])
demos = [demos[i] for i in inds]
# extract length of each trajectory in the file
traj_lengths = []
action_min = np.inf
action_max = -np.inf
joint_pos_min = np.inf
joint_pos_max = -np.inf
for ep in demos:
    traj_lengths.append(f["data/{}/actions".format(ep)].shape[0])
    action_min = min(action_min, np.min(f["data/{}/actions".format(ep)][()]))
    action_max = max(action_max, np.max(f["data/{}/actions".format(ep)][()]))
    joint_pos_min = min(joint_pos_min, np.min(f["data/{}/obs/robot0_joint_pos".format(ep)][()]))
    joint_pos_max = max(joint_pos_max, np.max(f["data/{}/obs/robot0_joint_pos".format(ep)][()]))
traj_lengths = np.array(traj_lengths)

# report statistics on the data
print("")
print("total transitions: {}".format(np.sum(traj_lengths)))
print("total trajectories: {}".format(traj_lengths.shape[0]))
print("traj length mean: {}".format(np.mean(traj_lengths)))
print("traj length std: {}".format(np.std(traj_lengths)))
print("traj length min: {}".format(np.min(traj_lengths)))
print("traj length max: {}".format(np.max(traj_lengths)))
print("action min: {}".format(action_min))
print("action max: {}".format(action_max))
print("")

ep = demos[10]
robot_joint_pos = f["data/{}/{}/{}".format(ep, "next_obs", "robot0_joint_pos")] 
gripper_qpos = f["data/{}/{}".format(ep, "actions")][:, -1:]  # gripper is the last action dim
# get the images
print("robot_joint_pos shape: ", robot_joint_pos.shape)
print("gripper_qpos shape: ", gripper_qpos.shape)
playback_actions = np.concatenate([robot_joint_pos, gripper_qpos], axis=-1)
print("playback_actions shape: ", playback_actions.shape)


total transitions: 9666
total trajectories: 200
traj length mean: 48.33
traj length std: 6.116461395284041
traj length min: 36
traj length max: 64
action min: -2.764532568641088
action max: 2.7911990332617242

robot_joint_pos shape:  (54, 7)
gripper_qpos shape:  (54, 1)
playback_actions shape:  (54, 8)


In [63]:
import mediapy as media
from diffusion_policy.env.robomimic.lift_joint_space import LiftJointSpace
from diffusion_policy.env_runner.robomimic_image_runner_joint_space import create_env
from diffusion_policy.env.robomimic.robomimic_image_wrapper import RobomimicImageWrapper
import robomimic.utils.file_utils as FileUtils
from mujoco_py import MjSimState


def playback(env: RobomimicImageWrapper, actions: np.array):
    vid = []
    print("playback actions shape: ", actions.shape)
    print("joint names", env.env.env.sim.model.joint_names)
    pbar = tqdm.tqdm(total=actions.shape[0])
    env.reset()
    env.env.env.sim.reset()
    env.env.env.sim.forward()

    for a in actions:
        obs, reward, done, info = env.step(a)
        # get the qpos names
        
        print("action", a[:-1])
        state = env.env.env.sim.get_state()
        state.qpos[:7] = a[:-1]
        state.qvel[:] = 0
        env.env.env.sim.set_state(state)
        env.env.env.sim.forward()
        for i in range(10):
            env.env.env.sim.forward()
            env.env.env.sim.step()
        print("new qpos: ", env.env.env.sim.data.qpos[:7])
        print("new ctrl: ", env.env.env.sim.data.ctrl[:-2])
        # print("new qpos: ", env.env.env.sim.data.qpos[:7])
        pbar.update()
        img = env.render()
        vid.append(img)
    pbar.close()
    return vid

env_meta = FileUtils.get_env_metadata_from_dataset(
    DATASET_PATH)
print("env_meta: ", env_meta)
# disable object state observation
env_meta['env_kwargs']['use_object_obs'] = False
env_meta['env_kwargs']['has_renderer'] = True
env_meta['env_kwargs']['has_offscreen_renderer'] = True
env_meta['env_kwargs']['controller_configs'] = {
    "type": "JOINT_POSITION",
    "input_max": 1,
    "input_min": -1,
    "output_max": 1,  #[0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.5],
    "output_min": -1,  #[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, -0.05, -0.5],
    "kp": 300,
    "damping_ratio": 10,
    "impedance_mode": "fixed",
    "kp_limits": [0, 300],
    "damping_ratio_limits": [0, 10],
    "qpos_limits": None,
    "interpolation": None,
    "ramp_ratio": 0.2,
    "input_type": "absolute"
  }
robomimic_env = create_env(
                env_meta=env_meta, 
                env_target=cfg.task.env_runner.env_target,
                shape_meta=cfg.task.shape_meta,
            )
#robomimic_env.env.hard_reset = False
import robomimic.utils.obs_utils as obs_utils
obs_utils.initialize_obs_utils_with_obs_specs({"obs": {"low_dim": ["robot0_joint_pos", "robot0_gripper_qpos"], "rgb": ["agentview_image", "robot0_eye_in_hand_image"]}})
robomimic_image_runner = RobomimicImageWrapper(
                env=robomimic_env,
                shape_meta=cfg.task.shape_meta,
                init_state=None,
                render_obs_key='agentview_image'
            )

env_meta:  {'env_name': 'Lift', 'type': 1, 'env_kwargs': {'has_renderer': False, 'has_offscreen_renderer': True, 'ignore_done': True, 'use_object_obs': True, 'use_camera_obs': True, 'control_freq': 20, 'controller_configs': {'type': 'OSC_POSE', 'input_max': 1, 'input_min': -1, 'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5], 'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5], 'kp': 150, 'damping': 1, 'impedance_mode': 'fixed', 'kp_limits': [0, 300], 'damping_limits': [0, 10], 'position_limits': None, 'orientation_limits': None, 'uncouple_pos_ori': True, 'control_delta': True, 'interpolation': None, 'ramp_ratio': 0.2}, 'robots': ['Panda'], 'camera_depths': False, 'camera_heights': 84, 'camera_widths': 84, 'reward_shaping': False, 'camera_names': ['agentview', 'robot0_eye_in_hand'], 'render_gpu_device_id': 0}}

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'robot0_joint_pos']
using obs modality: 

In [64]:
robomimic_image_runner.reset()
playback_actions_test = playback_actions.copy()
#playback_actions_test[:, 3] = 0
# get the joint min max for each joint
joint_min = np.min(playback_actions_test, axis=0)
joint_max = np.max(playback_actions_test, axis=0)
print("joint min: ", joint_min)
print("joint max: ", joint_max)

video = playback(robomimic_image_runner, playback_actions_test)
media.show_video(video, fps=30, height=256)

ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_pos not found, adding robot0_eef_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_quat not found, adding robot0_eef_quat to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_vel_lin not found, adding robot0_eef_vel_lin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_vel_ang not found, adding robot0_eef_vel_ang to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_gripper_qvel not found, adding robot0_gripper_qvel to mapping with assumed lo

  4%|▎         | 2/54 [00:01<00:41,  1.25it/s]

action [ 1.44126088e-02  1.94978823e-01 -8.31212117e-03 -2.65112180e+00
  2.21056321e-05  2.94716046e+00  7.93057381e-01]
new qpos:  [ 1.44209743e-02  1.95649556e-01 -8.28693531e-03 -2.65180559e+00
  6.01287592e-05  2.94755916e+00  7.93057953e-01]
new ctrl:  [ 0.33843187 -5.99075013  0.28561627 -0.05856765  0.29202209  1.68103125
  0.05187084]
action [ 1.44680791e-02  1.97579004e-01 -6.63349865e-03 -2.64486392e+00
 -1.85955837e-05  2.93928503e+00  7.94216722e-01]
new qpos:  [ 1.44704391e-02  1.97560535e-01 -6.63287864e-03 -2.64506615e+00
 -1.91773234e-05  2.93940735e+00  7.94279201e-01]
new ctrl:  [ 3.92934475e-02 -2.38265200e+01 -5.67219275e-02  9.67441751e+00
  9.43270181e-03  2.75185836e+00  3.56781743e-01]


  7%|▋         | 4/54 [00:02<00:18,  2.63it/s]

action [ 1.52443697e-02  2.03227023e-01 -2.93555882e-03 -2.63368806e+00
 -4.50109955e-04  2.92677820e+00  7.99687527e-01]
new qpos:  [ 1.52460588e-02  2.03225811e-01 -2.93543931e-03 -2.63391621e+00
 -4.50698316e-04  2.92693088e+00  7.99757427e-01]
new ctrl:  [ 4.87574774e-02 -2.35772220e+01 -3.78218568e-02  9.37184016e+00
  7.55106863e-03  2.80918305e+00  3.88514947e-01]
action [ 1.64934648e-02  2.15943559e-01  6.94299819e-04 -2.61608116e+00
 -2.57453034e-03  2.91239557e+00  8.09896594e-01]
new qpos:  [ 1.64949081e-02  2.15943710e-01  6.94507460e-04 -2.61632897e+00
 -2.57521565e-03  2.91256471e+00  8.09975367e-01]
new ctrl:  [ 5.64149722e-02 -2.37120502e+01 -1.38298494e-02  9.19745160e+00
  4.60577860e-04  2.83298044e+00  4.25324568e-01]


...

 98%|█████████▊| 53/54 [00:12<00:00,  4.72it/s]

action [ 0.09646326  0.79391099 -0.05571775 -2.08835068 -0.02232115  3.00151583
  1.18916766]
new qpos:  [ 0.09650963  0.79414641 -0.05575416 -2.08934739 -0.0224052   3.0023671
  1.18970865]
new ctrl:  [  1.00512969 -25.8056741   -1.41775364  -1.4029632   -0.20511517
   4.43328786   2.23976283]
action [ 0.09671956  0.75905662 -0.0547568  -2.10745451 -0.02150882  2.98028892
  1.18608906]
new qpos:  [ 0.09676696  0.75929565 -0.05479441 -2.10846663 -0.02159285  2.98115394
  1.18663782]
new ctrl:  [  1.02785131 -25.35001605  -1.39187163  -1.36391156  -0.20234369
   4.45364435   2.27310028]


100%|██████████| 54/54 [00:12<00:00,  4.41it/s]


action [ 0.09758154  0.71492595 -0.05094016 -2.14951492 -0.02892069  2.9582945
  1.17551698]
new qpos:  [ 0.09762988  0.71515781 -0.05098195 -2.15052823 -0.0290085   2.95916692
  1.1760738 ]
new ctrl:  [  1.05279837 -24.92119802  -1.33879426  -1.29482291  -0.22012838
   4.46942778   2.30617879]


This browser does not support the video tag.
